# Plot Juno's trajectory.

In [ ]:
import pandas as pd
import datetime as dt
import numpy as np 
import matplotlib.pyplot as plt 

from matplotlib.ticker import AutoMinorLocator

from JunoPy.spice import InitSpiceKernels, get_juno_position

## Get the spacecraft position for the given times.

In [ ]:
start = dt.datetime(2016, 6, 20, 17, 30)
end   = dt.datetime(2019, 5, 31, 16)

time_array = pd.date_range(
        start=start, end=end, periods=10000).to_pydatetime().tolist()

InitSpiceKernels()
position, _ = get_juno_position(time_array)

In [ ]:
def split_position(position):
    """Given the position of the spacecraft, split the array for points insdie 10 deg cone."""
    latitude = np.abs(np.arctan2(position[:, 2], np.sqrt(position[:, 0]**2 + position[:, 1]**2))) * 180 / np.pi
    print(latitude)
    position_masked = np.ma.empty(position.shape)
    position_masked[:, 0] = np.ma.masked_where(latitude > 10, position[:, 0])
    position_masked[:, 1] = np.ma.masked_where(latitude > 10, position[:, 1])
    position_masked[:, 2] = np.ma.masked_where(latitude > 10, position[:, 2])
    
    print(np.ma.count_masked(position_masked))
    return position_masked
    
position_masked = split_position(position)

## Calculate the Joy et al. (2002) magnetopause and bow-shock model

In [ ]:
def calc_coeff_bs(p):
    A = -1.107 + 1.591 * p**(-0.25)
    B = -0.566 - 0.812 * p**(-0.25)
    C =  0.048 - 0.059 * p**(-0.25)
    D =  0.077 - 0.038 * p
    E = -0.874 - 0.299 * p
    F = -0.055 + 0.124 * p
    return A, B, C, D, E, F
    
def calc_coeff_mp(p):
    A = -0.134 + 0.488 * p**(-0.25)
    B = -0.581 - 0.225 * p**(-0.25)
    C = -0.186 - 0.016 * p**(-0.25)
    D = -0.014 + 0.096 * p
    E = -0.814 - 0.811 * p
    F = -0.050 + 0.168 * p
    return A, B, C, D, E, F

def calc_equatorial(p, N, ctype='MP'):
    
    if 'MP' in ctype:
        A, B, C, D, E, F = calc_coeff_mp(p)
    else:
        A, B, C, D, E, F = calc_coeff_bs(p)
        
    phi = np.linspace(-150, 150, N) * np.pi / 180
    
    a = C * np.cos(phi)**2 + E * np.sin(phi)**2 + F * np.sin(phi) * np.cos(phi)
    b = B * np.cos(phi) + D * np.sin(phi)
    c = A
    
    root1 = (-b + np.sqrt(b**2 - 4 * a * c)) / (2 * a)
    root2 = (-b - np.sqrt(b**2 - 4 * a * c)) / (2 * a)
    
    r = np.zeros(N)
    for i in range(N):
        if root1[i] > 0:
            r[i] = root1[i]
        else:
            r[i] = root2[i]
        
    x = r * 120 * np.cos(phi)
    y = r * 120 * np.sin(phi)    
    

    return x, y
    

x_MP, y_MP = calc_equatorial(0.383, 101, ctype='MP')
x_BS, y_BS = calc_equatorial(0.382, 101, ctype='BS')

## Read the MVA epochs file and get the spacecraft position for the corresponding time

In [ ]:
def read_csv_file():
    times = []
    durations = []
    PNevents = []
    NPevents = []
    with open('mva_epochs_selected.csv', 'r') as f:
        f.readline()
        
        for i, line in enumerate(f):    

            times.append(dt.datetime.strptime(line.split(',')[0], '%Y-%jT%H:%M:%S'))
            durations.append(float(line.split(',')[2]))
            
            if 'PN' in line.split(',')[3]:
                PNevents.append(i)
            else:
                NPevents.append(i)
            
    return times, durations, PNevents, NPevents
            
times_mva, durations, PNevents, NPevents = read_csv_file()
position_mva, _ = get_juno_position(times_mva)

## Plot the trajectory

In [ ]:
# fig = plt.figure(figsize=(8, 4), sharex=True, constrained_layout=False, dpi=600)
gs_kw = dict(height_ratios=[130, 65], hspace=0)
fig, ax = plt.subplots(2, 1, sharex=True, dpi=600, gridspec_kw=gs_kw, figsize=(6, 8))

ax1 = ax[0]
ax2 = ax[1]
    
ax1.plot(position[:, 0], position[:, 1], color=[0.3, 0.3, 0.3], zorder=1, alpha=0.5)
ax2.plot(position[:, 0], position[:, 2], color=[0.3, 0.3, 0.3], zorder=1, alpha=0.5)
# ax1.plot(position_masked[:, 0], position_masked[:, 1], color=[0.3, 0.3, 0.3], zorder=1)
# ax2.plot(position_masked[:, 0], position_masked[:, 2], color=[0.3, 0.3, 0.3], zorder=1)
    
ax1.plot(x_BS, y_BS, color='k', zorder=0)
ax1.plot(x_MP, y_MP, color='k', ls='dashed', zorder=0)

ax1.xaxis.set_minor_locator(AutoMinorLocator())
ax1.yaxis.set_minor_locator(AutoMinorLocator())
ax2.xaxis.set_minor_locator(AutoMinorLocator())
ax2.yaxis.set_minor_locator(AutoMinorLocator())
ax1.grid('on', which='minor', lw=0.5, ls='dotted')
ax1.grid(color='k', ls='dashed', lw=0.25)
ax2.grid('on', which='minor', lw=0.5, ls='dotted')
ax2.grid(color='k', ls='dashed', lw=0.25)

ax1.set_xlabel('X [R]', weight='bold')
ax1.set_ylabel('Y [R]', weight='bold')
ax2.set_xlabel('X [R]', weight='bold')
ax2.set_ylabel('Z [R]', weight='bold')



# ax1.set_xlim([-105, 40])
ax1.set_ylim([-120, 10])

ax2.set_xlim([-105, 40])
ax2.set_ylim([-45, 20])

# ax1.set_adjustable("datalim", share=True)

ax1.set_aspect('equal', share=True)
ax2.set_aspect('equal', share=True)

sc1 = ax1.scatter(position_mva[:, 0], position_mva[:, 1], c=durations, zorder=2, marker='o', cmap='jet')
sc2 = ax2.scatter(position_mva[:, 0], position_mva[:, 2], c=durations, zorder=2, marker='o', cmap='jet')

bbox2 = ax2.get_position()
bbox1 = ax1.get_position()
cax = fig.add_axes([bbox2.x1, bbox2.y0, 0.016, bbox1.y1- bbox2.y0])


ax1.arrow(20, -115, 10, 0, color='k', head_width=3)
ax1.text(20, -110, 'To Sun')
ax1.text(30, -60, 'MP')
ax1.text(30, -90, 'BS')

ax2.text(-60, -40, 'Juno Trajectory', color=[0.3, 0.3, 0.3], alpha=0.5, weight='bold')

ax1.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))
ax2.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))

cbar2 = fig.colorbar(sc2, cax=cax);
cbar2.set_label('Peak-to-peak Duration (s)')
plt.subplots_adjust(wspace=0.3)
fig.savefig('out_images/TrajectoryLocationofEvent_selected.png')

In [ ]:
## 4. Plot position and color by polarity
nevents = len(durations)
polarity = np.zeros(nevents)
for i in range(nevents):
    polarity[i] = 1 if i in PNevents else 0
    


In [ ]:
# fig = plt.figure(figsize=(8, 4), sharex=True, constrained_layout=False, dpi=600)
gs_kw = dict(height_ratios=[130, 65], hspace=0)
fig, ax = plt.subplots(2, 1, sharex=True, dpi=600, gridspec_kw=gs_kw, figsize=(6, 8))

ax1 = ax[0]
ax2 = ax[1]
    
ax1.plot(position[:, 0], position[:, 1], color=[0.3, 0.3, 0.3], zorder=1, alpha=0.5)
ax2.plot(position[:, 0], position[:, 2], color=[0.3, 0.3, 0.3], zorder=1, alpha=0.5)
# ax1.plot(position_masked[:, 0], position_masked[:, 1], color=[0.3, 0.3, 0.3], zorder=1)
# ax2.plot(position_masked[:, 0], position_masked[:, 2], color=[0.3, 0.3, 0.3], zorder=1)
    
ax1.plot(x_BS, y_BS, color='k', zorder=0)
ax1.plot(x_MP, y_MP, color='k', ls='dashed', zorder=0)

ax1.xaxis.set_minor_locator(AutoMinorLocator())
ax1.yaxis.set_minor_locator(AutoMinorLocator())
ax2.xaxis.set_minor_locator(AutoMinorLocator())
ax2.yaxis.set_minor_locator(AutoMinorLocator())
ax1.grid('on', which='minor', lw=0.5, ls='dotted')
ax1.grid(color='k', ls='dashed', lw=0.25)
ax2.grid('on', which='minor', lw=0.5, ls='dotted')
ax2.grid(color='k', ls='dashed', lw=0.25)

ax1.set_xlabel('X [R]', weight='bold')
ax1.set_ylabel('Y [R]', weight='bold')
ax2.set_xlabel('X [R]', weight='bold')
ax2.set_ylabel('Z [R]', weight='bold')



# ax1.set_xlim([-105, 40])
ax1.set_ylim([-120, 10])

ax2.set_xlim([-105, 40])
ax2.set_ylim([-45, 20])

# ax1.set_adjustable("datalim", share=True)

ax1.set_aspect('equal', share=True)
ax2.set_aspect('equal', share=True)

sc1 = ax1.scatter(position_mva[PNevents, 0], position_mva[PNevents, 1], color='red', label='+/-', zorder=2)
sc2 = ax1.scatter(position_mva[NPevents, 0], position_mva[NPevents, 1], color='blue', label='-/+', zorder=2)
sc3 = ax2.scatter(position_mva[PNevents, 0], position_mva[PNevents, 2], color='red', label='+/-', zorder=2)
sc4 = ax2.scatter(position_mva[NPevents, 0], position_mva[NPevents, 2], color='blue', label='-/+', zorder=2)

ax1.legend()
ax2.legend()
bbox2 = ax2.get_position()
bbox1 = ax1.get_position()
# cax = fig.add_axes([bbox2.x1, bbox2.y0, 0.016, bbox1.y1- bbox2.y0])


ax1.arrow(20, -115, 10, 0, color='k', head_width=3)
ax1.text(20, -110, 'To Sun')
ax1.text(30, -60, 'MP')
ax1.text(30, -90, 'BS')
ax2.text(-60, -40, 'Juno Trajectory', color=[0.3, 0.3, 0.3], alpha=0.5, weight='bold')
ax1.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))
ax2.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))

plt.suptitle('Polarity of events', y=0.91)
plt.subplots_adjust(wspace=0.3)
fig.savefig('out_images/TrajectoryLocationofEvent_polarity_selected.png')

## Plot the trajectory only. 

In [ ]:
# fig = plt.figure(figsize=(8, 4), sharex=True, constrained_layout=False, dpi=600)
gs_kw = dict(height_ratios=[130, 65], hspace=0)
fig, ax = plt.subplots(2, 1, sharex=True, dpi=600, gridspec_kw=gs_kw, figsize=(6, 8))

ax1 = ax[0]
ax2 = ax[1]
    
ax1.plot(position[:, 0], position[:, 1], color=[0.3, 0.3, 0.3], zorder=1, alpha=0.5)
ax2.plot(position[:, 0], position[:, 2], color=[0.3, 0.3, 0.3], zorder=1, alpha=0.5)
    
ax1.plot(x_BS, y_BS, color='k', zorder=0)
ax1.plot(x_MP, y_MP, color='k', ls='dashed', zorder=0)

ax1.xaxis.set_minor_locator(AutoMinorLocator())
ax1.yaxis.set_minor_locator(AutoMinorLocator())
ax2.xaxis.set_minor_locator(AutoMinorLocator())
ax2.yaxis.set_minor_locator(AutoMinorLocator())
ax1.grid('on', which='minor', lw=0.5, ls='dotted')
ax1.grid(color='k', ls='dashed', lw=0.25)
ax2.grid('on', which='minor', lw=0.5, ls='dotted')
ax2.grid(color='k', ls='dashed', lw=0.25)

ax1.set_xlabel('X [R]', weight='bold')
ax1.set_ylabel('Y [R]', weight='bold')
ax2.set_xlabel('X [R]', weight='bold')
ax2.set_ylabel('Z [R]', weight='bold')



# ax1.set_xlim([-105, 40])
ax1.set_ylim([-120, 10])

ax2.set_xlim([-105, 40])
ax2.set_ylim([-45, 20])

# ax1.set_adjustable("datalim", share=True)

ax1.set_aspect('equal', share=True)
ax2.set_aspect('equal', share=True)

# sc1 = ax1.scatter(position_mva[:, 0], position_mva[:, 1], c=durations, zorder=2, marker='o', cmap='jet')
# sc2 = ax2.scatter(position_mva[:, 0], position_mva[:, 2], c=durations, zorder=2, marker='o', cmap='jet')

# bbox2 = ax2.get_position()
# bbox1 = ax1.get_position()
# cax = fig.add_axes([bbox2.x1, bbox2.y0, 0.016, bbox1.y1- bbox2.y0])

specificTime = dt.datetime.strptime('2018-085T14:53:45', '%Y-%jT%H:%M:%S')
specificTimeEnd = specificTime + dt.timedelta(hours=2)
specificPos, _ = get_juno_position([specificTime, specificTimeEnd])
print(specificPos)

ax1.scatter(specificPos[:, 0], specificPos[:, 1])
ax2.scatter(specificPos[:, 0], specificPos[:, 2])

ax1.arrow(20, -115, 10, 0, color='k', head_width=3)
ax1.text(20, -110, 'To Sun')
ax1.text(30, -60, 'MP')
ax1.text(30, -90, 'BS')

ax1.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))
ax2.add_artist(plt.Circle((0, 0), 1.0, color='k', zorder=0))

# cbar2 = fig.colorbar(sc2, cax=cax);
# cbar2.set_label('Peak-to-peak Duration (s)')
plt.subplots_adjust(wspace=0.3)
fig.savefig('out_images/Trajectory.png')